# Day 08: Exception Handling for Data Programs

**Dataset:** `Student_Performance_Dataset.csv`

This notebook learns how to handle common Python errors using `try`, `except`, `else`, and
`finally` while processing data, using real values pulled directly from the Student
Performance dataset, including a genuinely invalid `math_score` entry and thousands of
missing values.

## 1. Import Libraries and Load the Dataset

In [ ]:
import pandas as pd

# Load the dataset
df = pd.read_csv("../datasets/Student_Performance_Dataset.csv")

# Preview the data
df.head()

,roll_no,gender,race_ethnicity,parental_level_of_education,lunch,test_preparation_course,math_score,reading_score,writing_score,science_score,total_score,grade
0,std-01,male,group D,some college,1.0,1.0,89,38.0,85.0,26.0,238.0,C
1,std-02,male,group B,high school,1.0,0.0,65,100.0,67.0,96.0,328.0,A
2,std-03,male,group C,master's degree,1.0,0.0,10,99.0,97.0,58.0,264.0,B
3,std-04,male,group D,some college,1.0,1.0,22,51.0,41.0,84.0,198.0,D
4,std-05,male,group C,some college,0.0,1.0,26,58.0,64.0,65.0,213.0,C


In [ ]:
# Basic info about the dataset
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 12 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   roll_no                      9999 non-null   str    
 1   gender                       9982 non-null   str    
 2   race_ethnicity               9977 non-null   str    
 3   parental_level_of_education  9978 non-null   str    
 4   lunch                        9976 non-null   float64
 5   test_preparation_course      9977 non-null   float64
 6   math_score                   9976 non-null   str    
 7   reading_score                9975 non-null   float64
 8   writing_score                9976 non-null   float64
 9   science_score                9977 non-null   float64
 10  total_score                  9981 non-null   float64
 11  grade                        9997 non-null   str    
dtypes: float64(6), str(6)
memory usage: 937.6 KB


In [ ]:
# Count of missing values per column - this dataset has real missing data to handle
df.isna().sum()

roll_no                         1
gender                         18
race_ethnicity                 23
parental_level_of_education    22
lunch                          24
test_preparation_course        23
math_score                     24
reading_score                  25
writing_score                  24
science_score                  23
total_score                    19
grade                           3
dtype: int64

**Pull the `math_score` column into a plain Python list, to work with below**

In [ ]:
math_scores = list(df["math_score"])
print(math_scores[:10])
print("Total values:", len(math_scores))

['89', '65', '10', '22', '26', '40', '34', '25', '28', '71']
Total values: 10000


## 2. Examples of Common Python Exceptions

Each example below deliberately triggers a specific exception and catches it with `try` /
`except`, printing what went wrong.

**`ValueError` — trying to convert an invalid real value to a number**

In [ ]:
# Row std-2866 has a genuinely corrupted math_score value in the real dataset
bad_value = df.loc[df["roll_no"] == "std-2866", "math_score"].values[0]
print("Raw value:", repr(bad_value))

try:
    converted = float(bad_value)
except ValueError as e:
    print("ValueError caught:", e)

Raw value: '\\t41'
ValueError caught: could not convert string to float: '\\t41'


**`ValueError` — trying to convert a missing value (`NaN`) to an integer**

In [ ]:
missing_reading_score = df["reading_score"].iloc[df["reading_score"].isna().idxmax()]
print("Missing value:", missing_reading_score)

try:
    converted = int(missing_reading_score)
except ValueError as e:
    print("ValueError caught:", e)

Missing value: nan
ValueError caught: cannot convert float NaN to integer


**`TypeError` — trying to add incompatible types together**

In [ ]:
try:
    result = missing_reading_score + "points"   # float + str is not allowed
except TypeError as e:
    print("TypeError caught:", e)

TypeError caught: ufunc 'add' did not contain a loop with signature matching types (dtype('float64'), dtype('<U6')) -> None


**`ZeroDivisionError` — dividing by zero**

In [ ]:
number_of_courses_taken = 0

try:
    average = 100 / number_of_courses_taken
except ZeroDivisionError as e:
    print("ZeroDivisionError caught:", e)

ZeroDivisionError caught: division by zero


**`KeyError` — accessing a dictionary key that doesn't exist**

In [ ]:
student_record = {"roll_no": "std-01", "math_score": 89}

try:
    print(student_record["attendance"])
except KeyError as e:
    print("KeyError caught:", e)

KeyError caught: 'attendance'


**`IndexError` — accessing a list position that doesn't exist**

In [ ]:
short_list = [10, 20, 30]

try:
    print(short_list[10])
except IndexError as e:
    print("IndexError caught:", e)

IndexError caught: list index out of range


## 3. A Program Using `try`/`except`

A small program that attempts to convert a raw score to a float and reports whether it
succeeded, catching only the specific exception it expects (`ValueError`) rather than every
possible error.

In [ ]:
def process_score(raw_value):
    try:
        score = float(raw_value)
    except ValueError:
        print(f"Skipped invalid value: {raw_value!r}")
        return None
    else:
        # else only runs if no exception occurred - the conversion succeeded
        print(f"Processed value: {score}")
        return score


# Test it on a normal value and on the known-bad value
process_score(math_scores[0])
process_score(df.loc[df["roll_no"] == "std-2866", "math_score"].values[0])

Processed value: 89.0
Skipped invalid value: '\\t41'


## 4. A Data Conversion Program That Safely Handles Invalid Values

This program converts every value in `math_scores` to a float, keeping a count of
successes and failures. It uses `finally` to guarantee that every row is counted as
processed, whether the conversion succeeded or not.

In [ ]:
cleaned_scores = []
success_count = 0
failure_count = 0

for value in math_scores:
    try:
        if pd.isna(value):
            raise ValueError("missing value")
        cleaned_value = float(value)
    except ValueError:
        cleaned_scores.append(None)
        failure_count += 1
    else:
        cleaned_scores.append(cleaned_value)
        success_count += 1
    finally:
        pass  # in a real program, this is where you'd log/update a progress counter regardless of outcome

print("Successfully converted:", success_count)
print("Failed to convert:", failure_count)
print("First 15 cleaned scores:", cleaned_scores[:15])

Successfully converted: 9975
Failed to convert: 25
First 15 cleaned scores: [89.0, 65.0, 10.0, 22.0, 26.0, 40.0, 34.0, 25.0, 28.0, 71.0, 55.0, None, 29.0, 18.0, 72.0]


**Using `finally` for guaranteed cleanup**

Here, `finally` is used to print a status message after every single attempt, whether it
succeeded or failed, which is useful for logging or releasing a resource that must always run.

In [ ]:
def safe_convert(raw_value):
    try:
        if pd.isna(raw_value):
            raise ValueError("missing value")
        return float(raw_value)
    except ValueError as e:
        print(f"  Could not convert {raw_value!r}: {e}")
        return None
    finally:
        print("  Finished attempt.")


print("Attempt 1:")
safe_convert(math_scores[0])

print("Attempt 2:")
safe_convert(df.loc[df["roll_no"] == "std-2866", "math_score"].values[0])

Attempt 1:
  Finished attempt.
Attempt 2:
  Could not convert '\\t41': could not convert string to float: '\\t41'
  Finished attempt.


## Outcome

This notebook practiced handling common Python exceptions, namely `ValueError`, `TypeError`,
`ZeroDivisionError`, `KeyError`, and `IndexError`, using `try`/`except`, including on a
genuinely corrupted `math_score` value and real missing data found in the
`Student_Performance_Dataset.csv` file. I wrote a `process_score()` program that uses
`try`/`except`/`else` to handle invalid conversions, and a data conversion program that
processed the entire `math_score` column, safely skipping invalid or missing values while
tracking how many conversions succeeded versus failed, using `finally` to guarantee a cleanup
step runs on every attempt regardless of outcome. This showed why exception handling matters
for real data: instead of a single bad value crashing the whole program, catching specific
exceptions lets the program skip or log the problem and keep processing the rest of the data.